# Lazada Uplift — Feature Engineering & Data Split

Notebook thứ hai của đề tài. Nhận đầu vào là kết luận từ `01_eda.ipynb`, làm ba việc:

1. **Làm sạch** — bỏ các cột rác mà EDA đã chỉ ra
2. **Kỹ nghệ đặc trưng** — tạo thêm đặc trưng dẫn xuất
3. **Chia dữ liệu** thành 4 tập theo thiết kế thực nghiệm

### Thiết kế chia dữ liệu

```
full_trainset (926.669 dòng, có bias)
        ├── Train        80%   -> huấn luyện mô hình
        └── Val          20%   -> tinh chỉnh siêu tham số

full_testset  (181.669 dòng, RCT)
        ├── RCT-select   50%   -> chọn quán quân giữa các mô hình
        └── RCT-holdout  50%   -> chạy MỘT lần, ra số báo cáo cuối
```

Lý do tách tập RCT làm đôi: nếu vừa dùng nó để **chọn** mô hình vừa dùng để **báo cáo** kết quả thì con số cuối cùng bị thổi phồng, vì đã chọn ra cái may mắn nhất trên chính tập đó. Tách đôi giúp con số báo cáo trung thực.

### Đầu ra

4 file parquet trong `dataset/` và một file mô tả đặc trưng trong `artifacts/`.

---
# Phần 1 — Nạp dữ liệu và kết quả EDA

### Bước 1 — Import thư viện

In [ ]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
DATA_DIR = os.path.join(ROOT, 'dataset')
FIG_DIR = os.path.join(ROOT, 'reports', 'figures')
ART_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

C_TREAT, C_CTRL, C_ACC = '#d95f02', '#2b5c8f', '#1b9e77'

def save_fig(name):
    plt.savefig(os.path.join(FIG_DIR, name + '.png'), bbox_inches='tight')

### Bước 2 — Đọc kết quả bàn giao từ notebook 01

Notebook 01 đã ghi ra file JSON chứa danh sách cột cần loại bỏ. Đọc lại thay vì gõ tay, để hai notebook không bao giờ lệch nhau.

In [ ]:
with open(os.path.join(ART_DIR, 'eda_columns_to_drop.json'), encoding='utf-8') as f:
    eda = json.load(f)

print('Cột hằng số        :', eda['cot_hang_so'])
print('Cột trùng lặp      :', list(eda['cot_trung_lap'].keys()))
print('Cột thừa tuyến tính:', eda['cot_thua_tuyen_tinh'])
print()
print('Đặc trưng sạch sau notebook 01:', len(eda['dac_trung_sach']))

### Bước 3 — Nạp dữ liệu và áp lại phân chia của notebook 01

Notebook 01 đã chia dữ liệu **trước khi** nhìn vào nó, và ghi kết quả ra `split_assignment.parquet`. Ở đây đọc lại đúng file đó thay vì chia lại — chia hai lần bằng hai đoạn code khác nhau là mầm mống của sai lệch âm thầm.

In [ ]:
full_tr = pd.read_csv(os.path.join(DATA_DIR, 'full_trainset.csv'))
full_te = pd.read_csv(os.path.join(DATA_DIR, 'full_testset.csv'))

FEATURES = [f'f{i}' for i in range(83)]
for df in (full_tr, full_te):
    df[FEATURES] = df[FEATURES].astype('float32')

assignment = pd.read_parquet(os.path.join(ART_DIR, 'split_assignment.parquet'))
split_of = dict(zip(assignment['data_id'], assignment['split']))

full_tr['split'] = full_tr['data_id'].map(split_of)
full_te['split'] = full_te['data_id'].map(split_of)

assert full_tr['split'].notna().all() and full_te['split'].notna().all(), \
    'Có dòng không tìm thấy trong bảng phân chia — chạy lại notebook 01 trước.'

display(pd.concat([full_tr['split'], full_te['split']]).value_counts().to_frame('Số dòng'))

### Bước 4 — Tách phần được phép nhìn

Mọi phân tích trong notebook này chỉ dựa trên `train` và `rct_select`. Hai tập `val` và `rct_holdout` vẫn được biến đổi và ghi ra file, nhưng **không có thống kê nào được tính trên chúng**.

In [ ]:
train_raw = full_tr[full_tr['split'] == 'train'].reset_index(drop=True)
rct_raw = full_te[full_te['split'] == 'rct_select'].reset_index(drop=True)

print(f'train      : {train_raw.shape[0]:>8,} dòng  <- phân tích trên tập này')
print(f'rct_select : {rct_raw.shape[0]:>8,} dòng  <- và tập này')
print()
print('val và rct_holdout: chỉ đi qua bước biến đổi, không được phân tích.')

### Bước 5 — Nhắc lại các chỉ số chính từ EDA

In [ ]:
chi_so = eda['chi_so_chinh']
pd.Series({
    'Tỉ lệ treat Train': f"{chi_so['ti_le_treat_train']*100:.2f}%",
    'Tỉ lệ treat RCT-select': f"{chi_so['ti_le_treat_rct_select']*100:.2f}%",
    'CVR Train': f"{chi_so['cvr_train']*100:.2f}%",
    'ATE ngây thơ': f"{chi_so['ate_ngay_tho']*100:.3f} pp",
    'ATE thật (RCT)': f"{chi_so['ate_rct']*100:.3f} pp",
    'Mức thổi phồng': f"{chi_so['ty_le_thoi_phong']:.1f} lần",
    'Số đặc trưng SMD > 0,25': chi_so['so_dac_trung_smd_lon_025'],
}).to_frame('Giá trị')

---
# Phần 2 — Làm sạch dữ liệu

### Bước 6 — Bỏ cột hằng số và cột trùng lặp

Đây là 9 cột notebook 01 đã xác định: một cột hằng số và tám cột trùng lặp hoàn toàn với cột khác.

In [ ]:
base_features = eda['dac_trung_sach']
print('Bỏ', len(eda['cot_can_loai_bo']), 'cột ->', eda['cot_can_loai_bo'])
print('Còn lại:', len(base_features), 'đặc trưng')

### Bước 7 — Xử lý các cột nhị phân dư thừa

Notebook 01 đã phát hiện các quan hệ tuyến tính chính xác. Ở bước modeling, áp dụng một quy tắc chung chặt hơn cho toàn bộ cột nhị phân: hai cột được gom cùng nhóm nếu chúng giống nhau hoặc là phần bù của nhau ở ít nhất 99,999% số dòng train. Quy tắc có dung sai rất nhỏ để không giữ hai tín hiệu thực chất giống nhau chỉ vì vài dòng nhiễu.

Mỗi nhóm chỉ giữ cột có chỉ số nhỏ nhất; danh sách được tính hoàn toàn từ dữ liệu, không gõ tay tên feature.

In [ ]:
binary_candidates = [c for c in base_features if train_raw[c].nunique() == 2]
parent = {c: c for c in binary_candidates}

def find_root(c):
    while parent[c] != c:
        parent[c] = parent[parent[c]]
        c = parent[c]
    return c

def merge(a, b):
    ra, rb = find_root(a), find_root(b)
    if ra != rb:
        parent[rb] = ra

max_mismatch_rate = 1e-5
for i, a in enumerate(binary_candidates):
    for b in binary_candidates[i + 1:]:
        mismatch_same = (train_raw[a] != train_raw[b]).mean()
        mismatch_complement = ((train_raw[a] + train_raw[b]) != 1).mean()
        if min(mismatch_same, mismatch_complement) < max_mismatch_rate:
            merge(a, b)

redundant_groups = {}
for c in binary_candidates:
    redundant_groups.setdefault(find_root(c), []).append(c)
redundant_groups = [sorted(g, key=lambda c: int(c[1:]))
                    for g in redundant_groups.values() if len(g) > 1]
redundant_groups.sort(key=lambda g: int(g[0][1:]))
model_redundant = [c for group in redundant_groups for c in group[1:]]
assert set(eda['cot_thua_tuyen_tinh']).issubset(model_redundant)

print('Các nhóm cột nhị phân dư thừa:')
for group in redundant_groups:
    print(f'  {group} -> giữ {group[0]}')
print('Sẽ bỏ:', model_redundant)

Với mô hình cây quyết định, hai cột nhị phân giống nhau hoặc là phần bù gần như tuyệt đối cho cùng khả năng chia nhánh. Giữ một đại diện cho mỗi nhóm giúp feature importance rõ ràng hơn mà không làm mất tín hiệu đáng kể.

In [ ]:
model_base_features = [c for c in base_features if c not in model_redundant]

print('Bỏ thêm  :', model_redundant)
print('Còn lại  :', len(model_base_features), 'đặc trưng gốc')

### Bước 8 — Kiểm tra lại sau khi làm sạch

In [ ]:
kq = pd.DataFrame([
    ('Đặc trưng ban đầu', 83),
    ('Bỏ cột hằng số', -len(eda['cot_hang_so'])),
    ('Bỏ cột trùng lặp hoàn toàn', -len(eda['cot_trung_lap'])),
    ('Bỏ cột nhị phân dư thừa', -len(model_redundant)),
    ('Còn lại', len(model_base_features)),
], columns=['Bước', 'Số cột'])
kq

### Bước 9 — Xác nhận không còn cột thừa

In [ ]:
cm_check = train_raw[model_base_features].corr().abs()
pairs_check = cm_check.where(
    np.triu(np.ones(cm_check.shape), k=1).astype(bool)).stack().sort_values(ascending=False)

print('Số cột hằng số còn lại:', (train_raw[model_base_features].nunique() == 1).sum())
print('Số cặp có tương quan đúng bằng 1:', (pairs_check >= 1.0).sum())
print()
print('Ba cặp tương quan cao nhất còn lại:')
display(pairs_check.head(3).to_frame('|Tương quan|'))

Không còn cột hằng số, cũng không còn cặp nào tương quan **đúng bằng** 1.

Cặp cao nhất còn lại rất sát 1 nhưng vẫn khác nhau ở một số dòng, nên chúng mang thông tin không hoàn toàn trùng nhau. Giữ lại cả hai — mô hình cây chịu được đa cộng tuyến, và việc loại bỏ dựa trên một ngưỡng tự đặt sẽ khó biện minh hơn là dựa trên đẳng thức chính xác như ở bước 6.

Chỉ cần nhớ điều này khi đọc bảng feature importance ở notebook 04: giữa hai cột gần trùng nhau, việc mô hình chọn cột nào gần như là ngẫu nhiên.

> **Lưu ý về cách tính:** notebook 01 tìm cặp tương quan tuyệt đối trên một mẫu 150 nghìn dòng cho nhanh, còn ở đây tính trên toàn bộ 926 nghìn dòng. Một cặp có thể đạt đúng 1 trên mẫu nhưng chỉ còn 0,9999 trên toàn tập — đó là lý do bước này kiểm tra lại thay vì tin luôn kết quả cũ.

---
# Phần 3 — Kỹ nghệ đặc trưng

Dữ liệu ẩn danh nên không thể tạo đặc trưng theo hiểu biết nghiệp vụ. Thay vào đó tạo đặc trưng theo **quan hệ toán học giữa các cột** mà EDA đã chỉ ra: các cặp tương quan cao, và nhóm đặc trưng thiên vị mạnh nhất.

**Nguyên tắc quan trọng:** mọi đặc trưng ở đây đều tính **theo từng dòng**, không dùng thống kê gộp trên toàn tập. Nhờ vậy không có rò rỉ dữ liệu, và tính trước hay sau khi chia tập đều cho kết quả như nhau.

### Bước 10 — Phát hiện cấu trúc ẩn trong các cột nhị phân

Trước khi tạo đặc trưng mới, thử tìm hiểu xem gần 30 cột nhị phân kia thực chất là gì.

Giả thuyết: bảng phẳng này là kết quả của một luồng ETL, trong đó các **biến phân loại** đã bị mã hóa one-hot thành nhiều cột nhị phân. Nếu đúng, các cột thuộc cùng một biến gốc sẽ **loại trừ lẫn nhau** — mỗi dòng chỉ có đúng một cột bằng 1.

Kiểm tra bằng cách quét các cột nhị phân liền kề, gom dần cho tới khi tổng theo dòng luôn bằng 1.

In [ ]:
bin_cols = [c for c in model_base_features if train_raw[c].nunique() == 2]
print('Số cột nhị phân:', len(bin_cols))

onehot_groups = []
i = 0
while i < len(bin_cols):
    running = train_raw[bin_cols[i]].astype('float32').copy()
    j = i + 1
    while j < len(bin_cols) and not (running == 1).all():
        if (running > 1).any():
            break
        running = running + train_raw[bin_cols[j]].values
        j += 1
    if (running == 1).all() and (j - i) >= 2:
        onehot_groups.append(bin_cols[i:j])
        i = j
    else:
        i += 1

print(f'\nTìm thấy {len(onehot_groups)} nhóm one-hot:')
for k, g in enumerate(onehot_groups, 1):
    print(f'  Nhóm {k}: {len(g):>2} cột  ({g[0]} … {g[-1]})')

flag_cols = [c for c in bin_cols if not any(c in g for g in onehot_groups)]
print(f'\nCòn lại {len(flag_cols)} cờ nhị phân độc lập: {flag_cols}')

### Bước 11 — Xác nhận tính loại trừ lẫn nhau

In [ ]:
rows = []
for k, g in enumerate(onehot_groups, 1):
    s_tr = train_raw[g].sum(axis=1)
    s_te = rct_raw[g].sum(axis=1)
    rows.append({
        'Nhóm': f'Nhóm {k}',
        'Số cột': len(g),
        'Dải cột': f'{g[0]} … {g[-1]}',
        'Dòng vi phạm (Train)': int((s_tr != 1).sum()),
        'Dòng vi phạm (RCT-select)': int((s_te != 1).sum()),
    })

display(pd.DataFrame(rows))

Giả thuyết được xác nhận: cả bốn nhóm đều loại trừ lẫn nhau hoàn hảo trên tập Train, và gần như hoàn hảo trên tập Test.

Ngoại lệ duy nhất là vài bản ghi trong tập Test có **cả nhóm đều bằng 0** — tức biến phân loại gốc bị thiếu giá trị ở những dòng đó. Số lượng quá nhỏ so với quy mô tập dữ liệu nên không ảnh hưởng gì, nhưng đáng ghi nhận: đây là dấu vết cho thấy dữ liệu gốc **có** giá trị khuyết, chỉ là đã bị luồng ETL mã hóa thành toàn số 0 nên `isna()` ở notebook 01 không phát hiện được.

In [ ]:
g4 = onehot_groups[-1]
missing_rows = rct_raw[rct_raw[g4].sum(axis=1) == 0]
print(f'Số dòng thiếu giá trị ở nhóm cuối: {len(missing_rows)} / {len(rct_raw):,}')
if len(missing_rows):
    display(missing_rows[['data_id', 'is_treat', 'label'] + g4])

Nói cách khác, gần 30 cột nhị phân kia thực ra chỉ là **4 biến phân loại** (hai biến 3 mức, hai biến 10 mức) cộng thêm vài cờ độc lập. Đây là một phát hiện có giá trị vì hai lý do:

- **Cho thiết kế schema:** khi mô tả lại cấu trúc dữ liệu gốc, bốn biến phân loại đó nên là bốn cột, không phải 26 cột nhị phân rời rạc.
- **Cho kỹ nghệ đặc trưng:** ý tưởng "cộng tất cả cờ nhị phân lại" hóa ra vô nghĩa — mỗi nhóm one-hot luôn đóng góp đúng 1, nên tổng đó gần như là hằng số. Chỉ nên cộng các **cờ độc lập**.

Vẫn giữ nguyên dạng one-hot khi đưa vào mô hình. Cây quyết định xử lý one-hot tốt, còn nếu ép về dạng số thứ tự thì lại áp đặt một trật tự không có thật giữa các mức.

### Bước 12 — Kiểm tra cột gần như hằng số

Cột nhị phân mà 99,9% số dòng cùng một giá trị thì phần thiểu số quá ít để học được gì đáng tin.

In [ ]:
dominance = train_raw[model_base_features].apply(
    lambda s: s.value_counts(normalize=True).iloc[0]).sort_values(ascending=False)

print('Số cột có giá trị trội > 99,9%:', (dominance > 0.999).sum())
print('Số cột có giá trị trội > 99%  :', (dominance > 0.99).sum())
print()
display((dominance.head(8) * 100).round(3).to_frame('% dòng mang giá trị trội'))

Chỉ vài cột rơi vào diện này, và phần lớn trong số đó là **mức hiếm của một biến phân loại** chứ không phải cột rác — bỏ đi là mất luôn một mức của biến gốc.

Quyết định: **giữ lại tất cả**. Mô hình cây đơn giản là sẽ không chia nhánh trên chúng, nên chi phí gần như bằng 0, trong khi bỏ đi thì rủi ro mất thông tin về một phân khúc nhỏ.

### Bước 13 — Xem lại nhóm đặc trưng thiên vị nhất

Đây là nguyên liệu để tạo đặc trưng dẫn xuất.

In [ ]:
top_bias = [c for c in eda['top20_thien_vi'] if c in model_base_features]
print('Top đặc trưng thiên vị (còn lại sau làm sạch):')
print(top_bias[:12])

### Bước 14 — Nhóm 1: Tỉ lệ giữa các cặp tương quan cao

Khi hai cột tương quan gần 1, bản thân mỗi cột không nói thêm gì so với cột kia — nhưng **tỉ lệ giữa chúng** thì có. Nó thể hiện phần lệch khỏi quan hệ chung, thường mang ý nghĩa xu hướng tăng hay giảm.

Cộng 1 vào mẫu số để tránh chia cho 0.

In [ ]:
def add_ratio_features(df):
    df['fe_ratio_f1_f2'] = df['f1'] / (df['f2'] + 1.0)
    df['fe_ratio_f14_f16'] = df['f14'] / (df['f16'] + 1.0)
    df['fe_ratio_f9_f27'] = df['f9'] / (df['f27'] + 1.0)
    return df

print('Tương quan f1-f2  :', round(train_raw['f1'].corr(train_raw['f2']), 3))
print('Tương quan f14-f16:', round(train_raw['f14'].corr(train_raw['f16']), 3))
print('Tương quan f9-f27 :', round(train_raw['f9'].corr(train_raw['f27']), 3))

### Bước 15 — Nhóm 2: Đặc trưng tương tác

Tích của hai đặc trưng cho phép mô hình tuyến tính (LinearDML) nắm được quan hệ mà nếu chỉ dùng từng cột riêng lẻ thì không thấy được.

In [ ]:
def add_interaction_features(df):
    df['fe_inter_f9_f26'] = df['f9'] * df['f26']
    df['fe_inter_f14_f27'] = df['f14'] * df['f27']
    return df

### Bước 16 — Nhóm 3: Thống kê gộp theo dòng

Ý tưởng ban đầu là lấy trung bình và độ lệch chuẩn của nhóm 10 đặc trưng thiên vị nhất trên từng dòng, để tóm tắt "mức độ hoạt động chung" của khách hàng. Nhưng xem lại thang đo của chúng trước đã.

In [ ]:
top10_bias = top_bias[:10]
train_raw[top10_bias].describe().T[['mean', 'std', 'min', 'max']].round(2)

Thang đo lệch nhau quá xa — có cột chạy trong khoảng vài đơn vị, có cột lên tới hàng trăm. Lấy trung bình cộng của chúng thì kết quả gần như chỉ phản ánh **cột có thang đo lớn nhất**, các cột còn lại bị nhấn chìm. Đặc trưng như vậy không mang thêm thông tin gì so với chính cột lớn nhất đó.

Muốn dùng trung bình thì phải chuẩn hóa từng cột trước, mà việc đó cần tính thống kê trên tập huấn luyện rồi áp sang các tập khác — thêm phụ thuộc không cần thiết cho một đặc trưng phụ.

Thay bằng hai đặc trưng **không phụ thuộc thang đo**:

- Đếm số đặc trưng khác 0 trong nhóm top-10 — đo "độ rộng hoạt động" của khách hàng
- Tổng các **cờ nhị phân độc lập** — dùng danh sách `flag_cols` tìm được ở bước 9, không gộp các nhóm one-hot vào vì chúng luôn đóng góp đúng 1

In [ ]:
print('Top 10 thiên vị:', top10_bias)
print('Cờ độc lập     :', flag_cols)

def add_aggregate_features(df):
    df['fe_nonzero_top10'] = (df[top10_bias] != 0).sum(axis=1)
    df['fe_flag_sum'] = df[flag_cols].sum(axis=1)
    return df

### Bước 17 — Áp dụng cho cả hai tập

In [ ]:
def engineer_features(df):
    df = df.copy()
    df = add_ratio_features(df)
    df = add_interaction_features(df)
    df = add_aggregate_features(df)
    return df

# Áp cho TOÀN BỘ dữ liệu — các đặc trưng này tính theo dòng nên không có rò rỉ
full_tr_fe = engineer_features(full_tr)
full_te_fe = engineer_features(full_te)

# frame dùng để kiểm tra, vẫn chỉ là phần được phép nhìn
train_fe = full_tr_fe[full_tr_fe['split'] == 'train'].reset_index(drop=True)

ENGINEERED = ['fe_ratio_f1_f2', 'fe_ratio_f14_f16', 'fe_ratio_f9_f27',
              'fe_inter_f9_f26', 'fe_inter_f14_f27',
              'fe_nonzero_top10', 'fe_flag_sum']

MODEL_FEATURES = model_base_features + ENGINEERED

print('Đặc trưng gốc     :', len(model_base_features))
print('Đặc trưng dẫn xuất:', len(ENGINEERED))
print('Tổng đưa vào mô hình:', len(MODEL_FEATURES))

### Bước 18 — Kiểm tra các đặc trưng vừa tạo

In [ ]:
train_fe[ENGINEERED].describe().T.round(3)

In [ ]:
print('Số giá trị vô hạn:', np.isinf(train_fe[ENGINEERED].values).sum())
print('Số giá trị khuyết:', train_fe[ENGINEERED].isna().sum().sum())

Không có giá trị vô hạn hay khuyết thiếu — công thức chia đã cộng 1 vào mẫu số nên an toàn.

---
# Phần 4 — Tách bốn tập

Việc chia đã làm xong ở notebook 01, trước khi bất kỳ ai nhìn vào dữ liệu. Ở đây chỉ cắt các frame đã qua kỹ nghệ đặc trưng theo đúng phân chia đó.

### Bước 19 — Cắt theo cột `split`

In [ ]:
train_split = full_tr_fe[full_tr_fe['split'] == 'train'].reset_index(drop=True)
val_split = full_tr_fe[full_tr_fe['split'] == 'val'].reset_index(drop=True)
rct_select = full_te_fe[full_te_fe['split'] == 'rct_select'].reset_index(drop=True)
rct_holdout = full_te_fe[full_te_fe['split'] == 'rct_holdout'].reset_index(drop=True)

for nm, d in [('train', train_split), ('val', val_split),
              ('rct_select', rct_select), ('rct_holdout', rct_holdout)]:
    print(f'{nm:<12}: {len(d):>8,} dòng')

### Bước 20 — Vì sao phân chia phải stratify

Nhắc lại lý do, vì đây là chỗ dễ sai nhất: tỉ lệ mua hàng chỉ khoảng 2%, và tỉ lệ nhận voucher cũng lệch. Chia ngẫu nhiên thuần thì các tập con dễ lệch tỉ lệ, làm kết quả so sánh mất ý nghĩa.

Notebook 01 đã chia phân tầng theo `(is_treat, label)` — bốn tổ hợp, mỗi tổ hợp giữ đúng tỉ lệ trong mọi tập con. Bước sau kiểm chứng điều đó thực sự xảy ra.

In [ ]:
# dùng .to_numpy() vì hai frame có nhãn index trùng nhau, crosstab không căn được
strata_all = np.concatenate([
    (full_tr_fe['is_treat'].astype(str) + '_' + full_tr_fe['label'].astype(str)).to_numpy(),
    (full_te_fe['is_treat'].astype(str) + '_' + full_te_fe['label'].astype(str)).to_numpy(),
])
split_all = np.concatenate([full_tr_fe['split'].to_numpy(), full_te_fe['split'].to_numpy()])

ratio = pd.crosstab(pd.Series(split_all, name='Tập'),
                    pd.Series(strata_all, name='is_treat_label'),
                    normalize='index')
display(ratio.round(4))

print('Chênh lệch tỉ lệ lớn nhất giữa train và val       : '
      f"{(ratio.loc['train'] - ratio.loc['val']).abs().max()*100:.4f} pp")
print('Chênh lệch tỉ lệ lớn nhất giữa rct_select và holdout: '
      f"{(ratio.loc['rct_select'] - ratio.loc['rct_holdout']).abs().max()*100:.4f} pp")

### Bước 21 — Kiểm tra bốn tập có giữ đúng tỉ lệ không

In [ ]:
def describe_split(name, df):
    cvr_t = df.loc[df.is_treat == 1, 'label'].mean()
    cvr_c = df.loc[df.is_treat == 0, 'label'].mean()
    return {
        'Tập': name,
        'Số dòng': len(df),
        'Tỉ lệ treat': df['is_treat'].mean(),
        'Tỉ lệ mua': df['label'].mean(),
        'CVR treat': cvr_t,
        'CVR control': cvr_c,
        'Chênh lệch (pp)': (cvr_t - cvr_c) * 100,
    }

splits = {'Train': train_split, 'Val': val_split,
          'RCT-select': rct_select, 'RCT-holdout': rct_holdout}

split_info = pd.DataFrame([describe_split(k, v) for k, v in splits.items()])
split_info.round(4)

### Bước 22 — Xác nhận độ lệch nằm trong ngưỡng cho phép

In [ ]:
obs = split_info[split_info['Tập'].isin(['Train', 'Val'])]
rct = split_info[split_info['Tập'].isin(['RCT-select', 'RCT-holdout'])]

print('Chênh lệch tỉ lệ treat  — Train vs Val:',
      f"{abs(obs['Tỉ lệ treat'].diff().iloc[-1])*100:.4f} pp")
print('Chênh lệch tỉ lệ mua    — Train vs Val:',
      f"{abs(obs['Tỉ lệ mua'].diff().iloc[-1])*100:.4f} pp")
print()
print('Chênh lệch tỉ lệ treat  — select vs holdout:',
      f"{abs(rct['Tỉ lệ treat'].diff().iloc[-1])*100:.4f} pp")
print('Chênh lệch tỉ lệ mua    — select vs holdout:',
      f"{abs(rct['Tỉ lệ mua'].diff().iloc[-1])*100:.4f} pp")
print()
print('Yêu cầu: dưới 0,10 pp')

---
# Phần 5 — Kiểm tra giả định Positivity

Các phương pháp doubly robust dựa trên ba giả định. Hai trong số đó không kiểm chứng được từ dữ liệu (unconfoundedness và SUTVA), nhưng **positivity thì kiểm được**.

**Positivity** đòi hỏi: với mọi loại khách hàng, xác suất được phát voucher phải nằm hẳn trong khoảng (0, 1) — không có nhóm nào chắc chắn được phát hoặc chắc chắn không được phát.

Nếu vi phạm, công thức doubly robust có mẫu số tiến về 0 làm trọng số bùng nổ, và ước lượng trở nên vô nghĩa.

Cách kiểm: huấn luyện một mô hình dự đoán **ai được phát voucher**, rồi xem phân phối xác suất dự đoán của hai nhóm có chồng lấn nhau không.

### Bước 23 — Huấn luyện mô hình propensity

Dùng cross-fitting 3 fold để mỗi dòng đều được chấm bởi mô hình **chưa từng nhìn thấy nó**. Nếu chấm ngay trên dữ liệu đã huấn luyện, mô hình sẽ tách hai nhóm rõ hơn thực tế do overfitting, khiến ta tưởng nhầm là vi phạm positivity.

Lấy mẫu 300 nghìn dòng cho nhanh — đây chỉ là bước chẩn đoán, mô hình propensity thật sẽ được xây lại trong DR-Learner ở notebook sau.

In [ ]:
diag = train_split.sample(n=min(300_000, len(train_split)), random_state=SEED)
X_diag = diag[MODEL_FEATURES].values
W_diag = diag['is_treat'].values

prop_model = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05,
                                max_depth=5, random_state=SEED, verbose=-1)

e_hat = cross_val_predict(prop_model, X_diag, W_diag, cv=3,
                          method='predict_proba')[:, 1]

print('ROC-AUC của mô hình propensity:', round(roc_auc_score(W_diag, e_hat), 4))

AUC cao xác nhận lại kết luận của EDA: chỉ nhìn vào đặc trưng là đoán được khá chắc ai sẽ được phát voucher. Đó chính là thiên vị chọn lọc, đo bằng một cách khác.

Lưu ý AUC cao **không** đồng nghĩa với vi phạm positivity. Positivity chỉ hỏng khi có nhóm mà xác suất tiến sát 0 hoặc 1. Bước sau sẽ kiểm điều đó.

### Bước 24 — Phân phối propensity score của hai nhóm

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 60)
ax.hist(e_hat[W_diag == 0], bins=bins, alpha=0.55, color=C_CTRL,
        label='Control (W=0)', density=True)
ax.hist(e_hat[W_diag == 1], bins=bins, alpha=0.55, color=C_TREAT,
        label='Treatment (W=1)', density=True)
ax.axvline(0.01, color='red', ls='--', lw=1.2)
ax.axvline(0.99, color='red', ls='--', lw=1.2, label='Ngưỡng cắt [0,01 – 0,99]')
ax.set_xlabel('Propensity score — xác suất được phát voucher')
ax.set_ylabel('Mật độ')
ax.set_title('Kiểm tra Positivity: hai nhóm có chồng lấn nhau không?')
ax.legend()
plt.tight_layout()
save_fig('10_positivity_propensity')
plt.show()

### Bước 25 — Đo mức chồng lấn bằng số

In [ ]:
rows = []
for lo, hi in [(0.01, 0.99), (0.05, 0.95), (0.10, 0.90)]:
    outside = ((e_hat < lo) | (e_hat > hi)).mean()
    rows.append({'Ngưỡng cắt': f'[{lo} – {hi}]',
                 'Tỉ lệ bị cắt': f'{outside*100:.2f}%',
                 'Số dòng bị cắt': int(outside * len(e_hat))})

display(pd.DataFrame(rows))

print('Propensity nhỏ nhất:', round(e_hat.min(), 5))
print('Propensity lớn nhất:', round(e_hat.max(), 5))

### Bước 26 — Những dòng nào thực sự nguy hiểm?

Tỉ lệ nằm ngoài ngưỡng ở bước trên nghe có vẻ lớn, nhưng **không phải dòng cực trị nào cũng gây hại**. Nhìn vào công thức doubly robust sẽ rõ:

$$\underbrace{\frac{W(Y - \hat{\mu}_1)}{\hat{e}}}_{\text{chỉ áp dụng khi } W=1} \quad - \quad \underbrace{\frac{(1-W)(Y - \hat{\mu}_0)}{1 - \hat{e}}}_{\text{chỉ áp dụng khi } W=0}$$

Số hạng đầu chia cho $\hat{e}$, nhưng chỉ có hiệu lực với dòng **được phát voucher**. Một dòng control có $\hat{e}$ rất nhỏ thì hoàn toàn vô hại — số hạng đó bằng 0, còn mẫu số của số hạng sau là $1 - \hat{e} \approx 1$.

Dòng thực sự nguy hiểm chỉ gồm hai loại: **treated với $\hat{e}$ rất nhỏ**, và **control với $\hat{e}$ rất gần 1**.

In [ ]:
treated_low = ((W_diag == 1) & (e_hat < 0.01)).sum()
control_high = ((W_diag == 0) & (e_hat > 0.99)).sum()
nguy_hiem = treated_low + control_high

print(f'Dòng treated có e < 0,01   : {treated_low:>6,}  '
      f'({treated_low/(W_diag==1).sum()*100:.3f}% số dòng treated)')
print(f'Dòng control có e > 0,99   : {control_high:>6,}  '
      f'({control_high/(W_diag==0).sum()*100:.3f}% số dòng control)')
print()
ngoai_nguong = ((e_hat < 0.01) | (e_hat > 0.99)).mean()

print(f'Tổng dòng thực sự nguy hiểm: {nguy_hiem:,} / {len(e_hat):,} '
      f'= {nguy_hiem/len(e_hat)*100:.3f}%')
print()
print(f'So với {ngoai_nguong*100:.2f}% dòng nằm ngoài ngưỡng '
      f'-> phần lớn trong số đó hoàn toàn vô hại.')

### Bước 27 — Vùng chồng lấn của hai nhóm

In [ ]:
lo_t, hi_t = e_hat[W_diag == 1].min(), e_hat[W_diag == 1].max()
lo_c, hi_c = e_hat[W_diag == 0].min(), e_hat[W_diag == 0].max()

overlap_lo, overlap_hi = max(lo_t, lo_c), min(hi_t, hi_c)
inside = ((e_hat >= overlap_lo) & (e_hat <= overlap_hi)).mean()

print(f'Nhóm Treatment trải từ {lo_t:.4f} đến {hi_t:.4f}')
print(f'Nhóm Control   trải từ {lo_c:.4f} đến {hi_c:.4f}')
print(f'Vùng chồng lấn chung  : {overlap_lo:.4f} — {overlap_hi:.4f}')
print(f'Tỉ lệ dòng nằm trong vùng chồng lấn: {inside*100:.2f}%')

### Tóm tắt Phần 5

Hai nhóm chồng lấn trên gần như toàn bộ dải propensity, và không có dòng nào chạm sát 0 hay 1 tuyệt đối — **giả định positivity được thỏa mãn**, các phương pháp doubly robust áp dụng được.

Cần nói rõ một con số dễ gây hiểu nhầm: khoảng **16% bản ghi nằm ngoài ngưỡng `[0,01 – 0,99]`**. Nghe thì lớn, nhưng bước 23 cho thấy gần như toàn bộ trong số đó là **dòng control có propensity thấp** — hoàn toàn vô hại, vì số hạng chia cho $\hat{e}$ chỉ có hiệu lực với dòng được phát voucher. Số dòng thực sự gây bùng nổ trọng số nhỏ hơn rất nhiều.

Con số 16% đó phản ánh đúng bản chất dữ liệu: chỉ 22% khách hàng được phát voucher, nên phần đông có xác suất nhận thấp là chuyện bình thường.

Cách xử lý: **cắt propensity score** về `[0,01 – 0,99]` thay vì loại bỏ dòng dữ liệu — giữ được toàn bộ mẫu mà vẫn chặn được trọng số cực đoan. Đổi lại chấp nhận một chút chệch nhỏ để giảm mạnh phương sai, đây là đánh đổi tiêu chuẩn trong tài liệu.

---
# Phần 6 — Xuất dữ liệu

### Bước 28 — Chọn cột cần lưu

Giữ `data_id` để truy vết ngược khi cần, cùng hai biến `is_treat`, `label` và toàn bộ đặc trưng đưa vào mô hình.

In [ ]:
KEEP_COLS = ['data_id', 'is_treat', 'label'] + MODEL_FEATURES
print('Số cột mỗi file:', len(KEEP_COLS))

### Bước 29 — Ghi 4 file parquet

In [ ]:
out_files = {
    'train': train_split, 'val': val_split,
    'rct_select': rct_select, 'rct_holdout': rct_holdout,
}

for name, df in out_files.items():
    path = os.path.join(DATA_DIR, f'{name}.parquet')
    df[KEEP_COLS].to_parquet(path, index=False)
    print(f'{name:<12} {df.shape[0]:>8,} dòng  ->  {path} '
          f'({os.path.getsize(path)/1024**2:.1f} MB)')

### Bước 30 — Ghi file mô tả đặc trưng

Notebook 03 và 04 sẽ đọc file này để biết cột nào là đặc trưng, tránh phải gõ lại danh sách.

In [ ]:
feature_info = {
    'dac_trung_goc': model_base_features,
    'dac_trung_dan_xuat': ENGINEERED,
    'tat_ca_dac_trung': MODEL_FEATURES,
    'cot_luu_trong_parquet': KEEP_COLS,
    'cau_truc_one_hot': {
        f'nhom_{k}': g for k, g in enumerate(onehot_groups, 1)
    },
    'co_nhi_phan_doc_lap': flag_cols,
    'cong_thuc_dan_xuat': {
        'fe_ratio_f1_f2': 'f1 / (f2 + 1)',
        'fe_ratio_f14_f16': 'f14 / (f16 + 1)',
        'fe_ratio_f9_f27': 'f9 / (f27 + 1)',
        'fe_inter_f9_f26': 'f9 * f26',
        'fe_inter_f14_f27': 'f14 * f27',
        'fe_nonzero_top10': f'đếm số cột khác 0 trong {top10_bias}',
        'fe_flag_sum': f'tổng của {len(flag_cols)} cờ nhị phân độc lập: {flag_cols}',
    },
    'chia_du_lieu': {
        k: {'so_dong': int(len(v)),
            'ti_le_treat': float(v['is_treat'].mean()),
            'ti_le_mua': float(v['label'].mean())}
        for k, v in out_files.items()
    },
    'propensity_chan_doan': {
        'roc_auc': float(roc_auc_score(W_diag, e_hat)),
        'e_min': float(e_hat.min()),
        'e_max': float(e_hat.max()),
        'ti_le_ngoai_khoang_001_099': float(((e_hat < 0.01) | (e_hat > 0.99)).mean()),
        'nguong_cat_khuyen_nghi': [0.01, 0.99],
    },
    'seed': SEED,
}

path = os.path.join(ART_DIR, 'feature_info.json')
with open(path, 'w', encoding='utf-8') as f:
    json.dump(feature_info, f, ensure_ascii=False, indent=2)

print('Đã ghi:', path)

### Bước 31 — Đọc lại kiểm tra

In [ ]:
check = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))

print('Shape:', check.shape)
print('Số cột đặc trưng:', len([c for c in check.columns if c not in ['data_id', 'is_treat', 'label']]))
print('Giá trị khuyết:', check.isna().sum().sum())
print()
display(check.head(3))

---
## Kết luận

### Đã làm

| Việc | Kết quả |
|---|---|
| Làm sạch | 83 đặc trưng còn lại 69 sau khi bỏ cột hằng số, trùng lặp và các tín hiệu nhị phân dư thừa |
| Kỹ nghệ đặc trưng | Thêm 7 đặc trưng dẫn xuất, tất cả tính theo dòng nên không rò rỉ dữ liệu |
| Chia dữ liệu | 4 tập, stratify theo `(is_treat, label)`, độ lệch tỉ lệ dưới 0,10 pp |
| Kiểm tra Positivity | Hai nhóm chồng lấn gần như toàn dải, giả định thỏa mãn |

### Đầu ra

```
dataset/train.parquet         -> huấn luyện mô hình
dataset/val.parquet           -> tinh chỉnh siêu tham số
dataset/rct_select.parquet    -> chọn quán quân
dataset/rct_holdout.parquet   -> chạy MỘT lần, ra số báo cáo cuối

artifacts/feature_info.json   -> danh sách đặc trưng và công thức dẫn xuất
```

### Quy tắc cần nhớ cho các notebook sau

- **Không chạm vào `rct_holdout.parquet`** cho tới bước đánh giá cuối cùng ở notebook 04
- Cắt propensity score về `[0,01 – 0,99]` trong mọi công thức doubly robust
- Không cần chuẩn hóa scale vì toàn bộ mô hình dùng cây quyết định

### Notebook tiếp theo

`03_model_tuning.ipynb` — nạp `train` và `val`, tinh chỉnh siêu tham số cho 6 mô hình bằng Optuna, chấm điểm trên tập Val bằng DR-score, ghi vết toàn bộ thí nghiệm lên MLflow.